# NeoStream | DR3 | TP1: Preparação dos dados

**Aluno:** Rene Francisco Ferreira Santos

Este notebook cobre os Exercícios 1 a 8 do TP1, que tratam da preparação de dados para o modelo de churn da NeoStream. A base `assinantes_neostream.csv` é gerada uma única vez na seção de setup e permanece na mesma pasta deste notebook, pois será reutilizada sem modificação nos TPs seguintes e no Assessment.

## Setup: geração da base `assinantes_neostream.csv`

O script abaixo roda uma única vez e gera, localmente, a base com cerca de 12 mil assinantes, aproximadamente 18% de churn e a coluna `ltv_alto`. O arquivo é determinístico: a semente fixa (`default_rng(7)`) garante que este e os próximos TPs trabalhem sobre exatamente a mesma base.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 12000

plano = rng.choice(["basico", "padrao", "premium"], size=n, p=[0.4, 0.4, 0.2])
metodo_pagamento = rng.choice(["cartao_credito", "boleto", "pix"], size=n, p=[0.55, 0.20, 0.25])
dispositivo_principal = rng.choice(["smart_tv", "celular", "computador", "tablet"], size=n, p=[0.35, 0.35, 0.20, 0.10])
tempo_assinatura_meses = np.round(np.clip(rng.exponential(scale=14, size=n), 0, 72), 1)
uso_medio_semanal_horas = np.round(np.clip(rng.normal(9, 4, n), 0, 40), 1)
numero_reclamacoes = rng.poisson(0.4, n)

# churn: menor tempo de assinatura, menor uso e mais reclamações aumentam a chance de cancelar
logit_churn = (
    -0.85
    - 0.04 * tempo_assinatura_meses
    - 0.06 * uso_medio_semanal_horas
    + 0.5 * numero_reclamacoes
    + rng.normal(0, 0.6, n)
)
prob_churn = 1 / (1 + np.exp(-logit_churn))
churn = (rng.uniform(0, 1, n) < prob_churn).astype(int)

# ltv_alto: assinantes de alto valor formam uma região (não um corte linear simples) no espaço
# tempo_assinatura x uso_medio_semanal, com alguma sobreposição de classes nas bordas
raio = np.sqrt(((tempo_assinatura_meses - 40) / 20) ** 2 + ((uso_medio_semanal_horas - 14) / 8) ** 2)
ltv_alto = (raio < 1.0).astype(int)
inverter = rng.random(n) < 0.05
ltv_alto = np.where(inverter, 1 - ltv_alto, ltv_alto)

df = pd.DataFrame({
    "cliente_id": [f"AST-{i:05d}" for i in range(1, n + 1)],
    "plano": plano,
    "metodo_pagamento": metodo_pagamento,
    "dispositivo_principal": dispositivo_principal,
    "tempo_assinatura_meses": tempo_assinatura_meses,
    "uso_medio_semanal_horas": uso_medio_semanal_horas,
    "numero_reclamacoes": numero_reclamacoes,
    "churn": churn,
    "ltv_alto": ltv_alto,
})

df.to_csv("assinantes_neostream.csv", index=False)

In [2]:
base = pd.read_csv("assinantes_neostream.csv")

print(f"Shape da base gerada: {base.shape}")
print(f"Proporção de churn: {base['churn'].mean():.2%}")
print(f"Proporção de ltv_alto: {base['ltv_alto'].mean():.2%}")

base.head()

Shape da base gerada: (12000, 9)
Proporção de churn: 17.56%
Proporção de ltv_alto: 16.19%


,cliente_id,plano,metodo_pagamento,dispositivo_principal,tempo_assinatura_meses,uso_medio_semanal_horas,numero_reclamacoes,churn,ltv_alto
0,AST-00001,padrao,pix,smart_tv,14.3,10.1,0,0,0
1,AST-00002,premium,cartao_credito,smart_tv,0.8,5.6,0,0,0
2,AST-00003,padrao,cartao_credito,computador,7.3,6.8,0,0,0
3,AST-00004,basico,pix,computador,10.3,4.6,0,1,0
4,AST-00005,basico,boleto,smart_tv,2.6,5.8,0,0,0


# Exercício 1: Tipos de aprendizado e modos de treinamento

## Contexto

O time de produto da NeoStream trouxe quatro problemas de negócio. Antes de qualquer linha de código, cada problema precisa ser classificado por tipo de aprendizado e por modo de treinamento, para orientar a escolha de algoritmos e infraestrutura.

## Critérios de classificação

- **Tipo de aprendizado:** presença ou ausência de rótulos no problema descrito. É supervisionado quando existe um alvo conhecido a prever (mesmo que implícito) e não supervisionado quando não há rótulo algum, apenas estrutura a descobrir.
- **Modo de treinamento:** frequência de chegada de novos dados e necessidade de resposta em tempo real. É batch quando o modelo é treinado periodicamente com dados acumulados e online quando o modelo precisa ser atualizado a cada novo dado, com resposta imediata.

## Tabela consolidada

| Problema | Tipo de aprendizado | Modo de treinamento | Justificativa |
|---|---|---|---|
| Prever quais assinantes vão cancelar no próximo mês | Supervisionado | Batch | Existe rótulo histórico: a coluna `churn` (0 ou 1) registra o desfecho de assinantes passados, então o modelo aprende a relação entre features e um alvo conhecido. A previsão é mensal e não exige resposta em tempo real, então o modelo pode ser treinado em lote com os dados acumulados e atualizado periodicamente. |
| Agrupar assinantes por padrão de consumo sem rótulos prévios | Não supervisionado | Batch | O problema declara explicitamente a ausência de rótulos: não há um alvo a prever, o objetivo é descobrir grupos por similaridade (clustering). A segmentação é um produto periódico, não uma resposta em tempo real, então o modo batch é suficiente. |
| Recomendar séries em tempo real conforme o assinante assiste | Supervisionado | Online | O histórico de interação (o que o assinante assistiu, pausou ou avaliou) funciona como rótulo implícito: o modelo aprende a prever o próximo item a partir de interações passadas. A recomendação precisa reagir imediatamente a cada novo evento de visualização, então o modelo deve ser atualizado online, conforme cada dado novo chega. |
| Ajustar o preço de um plano promocional a cada reação do mercado | Supervisionado | Online | A reação do mercado a cada preço praticado (a demanda observada) funciona como rótulo: o modelo aprende a relação entre preço aplicado e resultado obtido. Como o ajuste deve ocorrer a cada nova reação, o treinamento precisa ser online. |

**Observação sobre o problema 4:** em uma formulação mais sofisticada, a precificação dinâmica é modelada como aprendizado por reforço (o preço é a ação e a reação do mercado é a recompensa). Dentro da dicotomia supervisionado/não supervisionado pedida no exercício, a reação do mercado cumpre o papel de rótulo, o que justifica a classificação como supervisionado.

# Exercício 2: Diagnóstico de bias e variância

## Contexto

Dois protótipos de churn foram treinados em uma base amostrada com o mesmo número de exemplos por classe (a referência de um chute aleatório é, portanto, 50% de acerto).

| Protótipo | Acurácia no treino | Acurácia no teste | Gap (treino - teste) | Diagnóstico |
|---|---|---|---|---|
| 1 | 98% | 61% | 37 pontos percentuais | Alta variância (overfitting) |
| 2 | 71% | 69% | 2 pontos percentuais | Viés relativamente alto (subajuste) |

## Diagnóstico de cada protótipo

- **Protótipo 1:** o modelo acerta quase tudo no treino e cai 37 pontos percentuais no teste. Esse gap gigante revela que ele decorou o conjunto de treino, incluindo ruído e peculiaridades que não se repetem em dados novos: é o caso clássico de alta variância (overfitting). Um modelo com 98% no treino e 61% no teste não serve para produção, porque o número que importa é o de teste.
- **Protótipo 2:** o gap de apenas 2 pontos percentuais indica que o modelo generaliza bem: o que ele aprende no treino se transfere quase integralmente para o teste. Porém, em uma base balanceada (referência aleatória de 50%), acertar 71% no próprio treino é um patamar baixo: o modelo é simples demais para capturar toda a estrutura dos dados. O diagnóstico é de viés relativamente alto (subajuste).

## Efeito da regularização no tradeoff bias-variância

A regularização é o controle de capacidade do modelo. Aumentá-la restringe o espaço de hipóteses, o que reduz a variância (o modelo deixa de perseguir ruído) mas aumenta o viés (o modelo pode deixar de capturar padrões reais); reduzi-la tem o efeito oposto.

- **No protótipo 1, o problema é variância:** aumentar a regularização reduz o gap entre treino e teste, porque o modelo fica impedido de memorizar o treino. O acerto de treino vai cair (o modelo abre mão de decorar ruído), mas o acerto de teste tende a subir ou a se manter, agora com custo de complexidade muito menor.
- **No protótipo 2, o problema é viés:** aumentar a regularização restringiria ainda mais um modelo que já é simples demais, derrubando os dois acertos e agravando o subajuste. Reduzir a regularização (ou oferecer features mais informativas e maior capacidade) é o que pode elevar o patamar de 71% sem sacrificar a generalização.

## Recomendação

- O **protótipo 1** deve receber **mais regularização** (em regressão logística, um `C` menor, ou seja, penalização L2 mais forte; em árvores, menos profundidade e mais amostras mínimas por folha).
- O **protótipo 2** deve receber **menos regularização** (`C` maior) ou ganhar features mais informativas e um modelo um pouco mais expressivo.
- Do ponto de vista do prazo da empresa, o protótipo 2 é a direção de partida mais segura: seu gap de 2 pontos mostra que ele generaliza, enquanto o protótipo 1, no estado atual, é exatamente o modelo que não generaliza e que a empresa não pode se dar ao luxo de treinar por mais uma semana. O caminho recomendado é partir do protótipo 2, aumentar sua capacidade gradualmente e monitorar cada passo por validação cruzada.

# Exercício 3: Divisão estratificada treino/teste

## Contexto

A base `assinantes_neostream.csv` tem cerca de 18% de churn. Antes de treinar qualquer classificador, os dados precisam ser separados de forma que a proporção de churn seja preservada nos dois conjuntos e que nenhuma etapa de pré-processamento ajustada antes da divisão vaze informação do teste para o treino.

## Tarefa

1. Carregar a base e separar `churn` como alvo.
2. Usar `train_test_split` com `stratify` e `test_size=0.2`.
3. Comprovar, com `value_counts(normalize=True)`, que a proporção de churn é aproximadamente igual nos dois conjuntos.
4. Explicar por que qualquer `fit` de scaler ou encoder deve ocorrer somente depois dessa divisão.

In [3]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("assinantes_neostream.csv")
X = df.drop(columns="churn")
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

X_train: (9600, 8) | X_test: (2400, 8)
y_train: (9600,) | y_test: (2400,)


In [4]:
proporcao_train = y_train.value_counts(normalize=True)
proporcao_test = y_test.value_counts(normalize=True)

print("Proporção de churn em y_train:")
print(proporcao_train)
print("\nProporção de churn em y_test:")
print(proporcao_test)

diferenca = abs(proporcao_train[1] - proporcao_test[1])
print(f"\nDiferença absoluta entre as proporções: {diferenca:.4%}")

Proporção de churn em y_train:
churn
0    0.824375
1    0.175625
Name: proportion, dtype: float64

Proporção de churn em y_test:
churn
0    0.824583
1    0.175417
Name: proportion, dtype: float64

Diferença absoluta entre as proporções: 0.0208%


## Resposta

**Registro da comparação:** a proporção de churn é de aproximadamente 17,6% nos dois conjuntos (valores exatos na saída acima), com diferença absoluta de 0,02 ponto percentual. A estratificação preservou a distribuição da classe alvo no treino e no teste, como pedido.

**Por que o `fit` só pode ocorrer depois da divisão:** qualquer `fit` de scaler ou encoder deve acontecer somente depois da divisão porque, se os parâmetros da transformação forem aprendidos com o conjunto de teste, informação do teste vaza para o treino e a avaliação deixa de representar dados não vistos (data leakage).

# Exercício 4: Pré-processamento heterogêneo com ColumnTransformer

## Contexto

A base mistura features numéricas (`tempo_assinatura_meses`, `uso_medio_semanal_horas`, `numero_reclamacoes`) e categóricas (`plano`, `metodo_pagamento`, `dispositivo_principal`). Aplicar `StandardScaler` ao DataFrame inteiro falha ao encontrar texto nas colunas categóricas. O pré-processamento heterogêneo precisa ficar organizado em um único objeto reutilizável, aplicável ao treino e a dados novos em produção.

## Tarefa

1. Construir um `ColumnTransformer` com `StandardScaler` nas numéricas e `OneHotEncoder` nas categóricas.
2. Ajustar apenas com `X_train` (nunca com `X_test`) e transformar os dois conjuntos.
3. Salvar o código em `preprocessamento_heterogeneo.py` e imprimir o shape da matriz resultante.

In [5]:
%%writefile preprocessamento_heterogeneo.py
"""Pré-processamento heterogêneo da base de assinantes da NeoStream.

Aplica StandardScaler às features numéricas e OneHotEncoder às categóricas
em um único objeto ColumnTransformer, reutilizável no treino e em produção.

Notas de design:
1. remainder="drop" descarta cliente_id (identificador sem sinal preditivo)
   e ltv_alto (rótulo do TP2, não é feature legítima para prever churn).
2. handle_unknown="ignore" faz o encoder ignorar categorias inéditas que
   surjam em produção, em vez de falhar.
3. sparse_threshold=0 força saída densa, simplificando inspeção e consumo.
"""

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

COLUNAS_NUMERICAS = [
    "tempo_assinatura_meses",
    "uso_medio_semanal_horas",
    "numero_reclamacoes",
]
COLUNAS_CATEGORICAS = ["plano", "metodo_pagamento", "dispositivo_principal"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), COLUNAS_NUMERICAS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORICAS),
    ],
    remainder="drop",
    sparse_threshold=0,
    verbose_feature_names_out=False,
)

Overwriting preprocessamento_heterogeneo.py


In [6]:
from preprocessamento_heterogeneo import preprocessor

# fit somente no treino; o teste (e qualquer dado novo) é apenas transformado
preprocessor.fit(X_train)

X_train_transformado = preprocessor.transform(X_train)
X_test_transformado = preprocessor.transform(X_test)

print("Shape de X_train original:", X_train.shape)
print("Shape de X_train transformado:", X_train_transformado.shape)
print("Shape de X_test transformado:", X_test_transformado.shape)
print("\nFeatures resultantes:")
print(list(preprocessor.get_feature_names_out()))

Shape de X_train original: (9600, 8)
Shape de X_train transformado: (9600, 13)
Shape de X_test transformado: (2400, 13)

Features resultantes:
['tempo_assinatura_meses', 'uso_medio_semanal_horas', 'numero_reclamacoes', 'plano_basico', 'plano_padrao', 'plano_premium', 'metodo_pagamento_boleto', 'metodo_pagamento_cartao_credito', 'metodo_pagamento_pix', 'dispositivo_principal_celular', 'dispositivo_principal_computador', 'dispositivo_principal_smart_tv', 'dispositivo_principal_tablet']


## Resposta

A matriz transformada tem 13 colunas: 3 numéricas escaladas pelo `StandardScaler` e 10 colunas dummies criadas pelo `OneHotEncoder` (3 planos + 3 métodos de pagamento + 4 dispositivos). O parâmetro `remainder="drop"` descarta as demais colunas: `cliente_id` não carrega sinal preditivo (é um identificador único) e `ltv_alto` é o rótulo usado no TP2, não uma feature legítima para prever churn, então incluí-la configuraria vazamento de dados. O `handle_unknown="ignore"` faz o encoder ignorar categorias inéditas em produção, em vez de falhar.

# Exercício 5: Pipeline de pré-processamento + classificação

## Contexto

O `ColumnTransformer` do Exercício 4 resolveu o pré-processamento, mas repetir manualmente os passos de transformação a cada lote novo de previsão cria risco de inconsistência entre treino e produção. Pré-processamento e classificador precisam ser encapsulados em um único objeto treinável e reutilizável.

## Tarefa

1. Construir um `Pipeline` sklearn com o `ColumnTransformer` do Exercício 4 e um `LogisticRegression` como estimador final.
2. Treinar com `pipeline.fit(X_train, y_train)` em uma única chamada, sem transformar os dados manualmente antes.
3. Gerar previsões com `pipeline.predict(X_test)` e salvar o código em `pipeline_classificacao.py`.
4. Explicar por que o `Pipeline` reduz o risco de data leakage em produção.

In [7]:
%%writefile pipeline_classificacao.py
"""Pipeline de classificação de churn da NeoStream.

Encapsula o pré-processamento heterogêneo (ColumnTransformer do Exercício 4)
e o classificador LogisticRegression em um único objeto treinável, aplicável
a dados crus tanto no treino quanto em produção.
"""

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from preprocessamento_heterogeneo import preprocessor

pipeline = Pipeline(
    steps=[
        ("preprocessamento", preprocessor),
        ("classificador", LogisticRegression(max_iter=2000, random_state=42)),
    ]
)

Overwriting pipeline_classificacao.py


In [8]:
from sklearn.metrics import accuracy_score

from pipeline_classificacao import pipeline

print(pipeline)

# única chamada: o Pipeline aplica o pré-processamento e treina o classificador
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
acuracia = accuracy_score(y_test, y_pred)
print(f"\nAcurácia no conjunto de teste: {acuracia:.4f}")

Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(sparse_threshold=0,
                                   transformers=[('num', StandardScaler(),
                                                  ['tempo_assinatura_meses',
                                                   'uso_medio_semanal_horas',
                                                   'numero_reclamacoes']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['plano', 'metodo_pagamento',
                                                   'dispositivo_principal'])],
                                   verbose_feature_names_out=False)),
                ('classificador',
                 LogisticRegression(max_iter=2000, random_state=42))])

Acurácia no conjunto de teste: 0.8250


## Resposta

Empacotar tudo em um `Pipeline` reduz o risco de data leakage porque o `fit` do Pipeline ajusta o `ColumnTransformer` apenas sobre os dados de treino e aplica a transformação já congelada ao teste ou a qualquer dado novo, sem deixar espaço para que parâmetros do pré-processamento sejam aprendidos em dados que não eram de treino. Além disso, o Pipeline inteiro é serializado como um único artefato, então produção executa exatamente a mesma transformação usada no treino, sem passos manuais que possam divergir entre os dois ambientes.

# Exercício 6: Busca em grade do hiperparâmetro C

## Contexto

O Pipeline do Exercício 5 funciona, mas foi treinado com os hiperparâmetros padrão do `LogisticRegression`. Antes de decidir o modelo de produção, é preciso verificar se outro valor do parâmetro de regularização `C` melhora a métrica de validação cruzada. Com orçamento computacional limitado, a busca deve ser sistemática (grid search), não manual.

## Tarefa

1. Definir a grade de valores de `C` (exemplo: `[0.01, 0.1, 1, 10, 100]`).
2. Configurar `GridSearchCV` com `cv=5` sobre o Pipeline e executar com `X_train, y_train`.
3. Reportar o melhor `C` (`best_params_`) e a métrica correspondente (`best_score_`).

In [9]:
from sklearn.model_selection import GridSearchCV

grade_c = {"classificador__C": [0.01, 0.1, 1, 10, 100]}

busca = GridSearchCV(
    pipeline,
    param_grid=grade_c,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
busca.fit(X_train, y_train)

print("Melhor combinação de hiperparâmetros:", busca.best_params_)
print(f"Melhor acurácia média na validação cruzada: {busca.best_score_:.4f}")

Melhor combinação de hiperparâmetros: {'classificador__C': 0.1}
Melhor acurácia média na validação cruzada: 0.8247


In [10]:
resultados = pd.DataFrame(busca.cv_results_)[
    ["param_classificador__C", "mean_test_score", "std_test_score"]
].rename(
    columns={
        "param_classificador__C": "C",
        "mean_test_score": "acurácia média (cv)",
        "std_test_score": "desvio padrão (cv)",
    }
)
resultados

,C,acurácia média (cv),desvio padrão (cv)
0,0.01,0.824583,0.000833
1,0.10,0.824687,0.001495
2,1.00,0.824583,0.001524
3,10.00,0.824687,0.001495
4,100.00,0.824687,0.001495


## Resposta

O melhor valor encontrado foi `C = 0.1` (saída acima), com acurácia média de validação cruzada de aproximadamente 0.8247. Os cinco valores da grade ficaram muito próximos entre si (diferenças na quarta casa decimal), o que indica que, para esta base e estas features, a escolha da regularização pouco altera o resultado e o `C` padrão (1) já era competitivo. O ponto central do exercício é o procedimento: com `GridSearchCV` a escolha é sistemática, auditável e reproduzível, no lugar de tentativa e erro manual, e a decisão final fica ancorada em uma métrica de validação cruzada em vez de um palpite.

# Exercício 7: Auditoria de data leakage na ordem das operações

## Contexto

Um estagiário ajustou o `StandardScaler` com o dataset completo (`X` antes da divisão) e só depois fez `train_test_split`. O código rodou sem erros, mas a ordem das operações contamina a avaliação do modelo. Esta seção audita o fluxo, identifica o ponto exato do vazamento e documenta a correção.

## Fluxo incorreto (auditado)

```python
# INCORRETO: scaler ajustado antes da divisão
scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)   # X contém as linhas que virarão o teste
X_train, X_test, y_train, y_test = train_test_split(X_escalado, y, test_size=0.2)
```

**Onde ocorre o vazamento:** na linha do `fit_transform`. A média e o desvio padrão de cada coluna numérica são calculados usando o conjunto inteiro, incluindo as linhas que depois viram o teste. Quando o modelo é avaliado em `X_test`, o teste não é mais um dado nunca visto: as features do teste foram transformadas com estatísticas que já conheciam o próprio teste. A métrica de teste reportada nesse fluxo é otimista demais porque a avaliação deixa de simular o cenário real de produção, em que os dados novos chegam depois que o scaler já foi fixado.

## Fluxo corrigido (ordem certa)

```python
# CORRETO: divisão estratificada antes de qualquer fit
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train_escalado = scaler.fit_transform(X_train)  # fit apenas no treino
X_test_escalado = scaler.transform(X_test)         # teste apenas transformado
```

## Evidência numérica do mecanismo

A célula abaixo compara os parâmetros do scaler ajustado com o dataset completo (fluxo incorreto) e ajustado só com o treino (fluxo correto). Os valores são diferentes, o que prova que as linhas de teste alteraram a transformação aplicada sobre elas.

In [11]:
from sklearn.preprocessing import StandardScaler

from preprocessamento_heterogeneo import COLUNAS_NUMERICAS

scaler_vazado = StandardScaler().fit(X[COLUNAS_NUMERICAS])
scaler_correto = StandardScaler().fit(X_train[COLUNAS_NUMERICAS])

comparacao = pd.DataFrame(
    {
        "média (fit no dataset completo, incorreto)": scaler_vazado.mean_,
        "média (fit só no treino, correto)": scaler_correto.mean_,
        "desvio padrão (fit no dataset completo, incorreto)": scaler_vazado.scale_,
        "desvio padrão (fit só no treino, correto)": scaler_correto.scale_,
    },
    index=COLUNAS_NUMERICAS,
)
comparacao.round(4)

,"média (fit no dataset completo, incorreto)","média (fit só no treino, correto)","desvio padrão (fit no dataset completo, incorreto)","desvio padrão (fit só no treino, correto)"
tempo_assinatura_meses,13.8582,13.8698,13.5660,13.5587
uso_medio_semanal_horas,9.0383,9.0239,3.9640,3.9716
numero_reclamacoes,0.3948,0.3960,0.6266,0.6255


## Justificativa técnica da diferença

No fluxo incorreto, os parâmetros do pré-processamento são uma função dos dados de teste, então parte da informação que deveria ser surpresa na avaliação já foi consumida na transformação. O efeito tende a inflar a métrica de teste, que deixa de ser uma estimativa honesta de generalização, e é tanto pior quanto mais informativa for a transformação: um scaler vaza pouco, mas imputação de médias, seleção de features ou target encoding vazam muito. No fluxo corrigido, o `fit` só enxerga o treino e o teste é tratado como um lote de produção novo, transformado com parâmetros congelados, o que reproduz fielmente o comportamento em produção. A correção adotada neste TP é exatamente o fluxo do Exercício 3, com estratificação na divisão.

# Exercício 8: Script final de produção

## Contexto

A entrega final do sprint é um único script que, a partir do dataset bruto `assinantes_neostream.csv`, produz um pipeline de classificação de churn pronto para receber novos dados em produção, já com os melhores hiperparâmetros encontrados por busca em grade. O script será revisado por outro engenheiro antes do deploy, então precisa estar organizado, comentado e sem etapas manuais fora do objeto Pipeline.

## Tarefa

1. Integrar em `pipeline_final_churn.py` as etapas dos Exercícios 3 a 6: divisão estratificada, `ColumnTransformer`, `Pipeline` com `LogisticRegression` e `GridSearchCV`.
2. Ajustar o Pipeline final com os melhores hiperparâmetros e treiná-lo sobre todo o `X_train`.
3. Incluir uma seção de comentários explicando cada decisão de design (por que estratificar, por que `ColumnTransformer`, por que `Pipeline` em vez de passos manuais).
4. Encerrar com um parágrafo explicando por que essa organização é mais segura para produção do que transformar os dados manualmente antes de treinar.

In [12]:
%%writefile pipeline_final_churn.py
"""Pipeline final de classificação de churn da NeoStream.

Script único e autocontido: carrega a base bruta assinantes_neostream.csv,
divide os dados de forma estratificada, otimiza o hiperparâmetro C do
LogisticRegression por GridSearchCV e entrega um Pipeline treinado e
serializado, pronto para receber novos dados em produção.

DECISÕES DE DESIGN
1. Divisão estratificada (stratify=y): o churn representa cerca de 18% da
   base. Um sorteio puramente aleatório poderia deixar treino e teste com
   proporções diferentes da classe alvo, enviesando o treinamento ou a
   avaliação. A estratificação garante a mesma proporção de churn nos dois
   conjuntos.
2. ColumnTransformer: a base mistura features numéricas e categóricas.
   Um único objeto aplica StandardScaler às numéricas e OneHotEncoder às
   categóricas, sem nenhum passo manual fora do Pipeline. O argumento
   handle_unknown="ignore" torna o encoder tolerante a categorias inéditas
   em produção, e remainder="drop" descarta cliente_id (sem sinal preditivo)
   e ltv_alto (rótulo do TP2, não é feature de churn).
3. Pipeline em vez de passos manuais: fit e predict passam a ser chamadas
   únicas sobre dados crus. Isso elimina a possibilidade de esquecer de
   transformar um conjunto, de ajustar um transformador no conjunto errado
   ou de divergir os parâmetros de pré-processamento entre treino e
   produção.
4. GridSearchCV com cv=5: a busca do melhor C é sistemática e barata
   (5 valores x 5 folds), compatível com o orçamento computacional limitado.
   Com refit=True (padrão), o GridSearchCV reajusta o melhor modelo em todo
   o X_train, então busca.best_estimator_ já é o modelo final a ser
   serializado.
"""

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CAMINHO_DADOS = "assinantes_neostream.csv"
CAMINHO_MODELO = "pipeline_final_churn.joblib"
COLUNAS_NUMERICAS = [
    "tempo_assinatura_meses",
    "uso_medio_semanal_horas",
    "numero_reclamacoes",
]
COLUNAS_CATEGORICAS = ["plano", "metodo_pagamento", "dispositivo_principal"]
RANDOM_STATE = 42


def carregar_dados(caminho):
    """Carrega a base bruta em um DataFrame."""
    return pd.read_csv(caminho)


def construir_preprocessador():
    """Monta o pré-processamento heterogêneo em um único objeto."""
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), COLUNAS_NUMERICAS),
            ("cat", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORICAS),
        ],
        remainder="drop",
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )


def construir_pipeline(preprocessador):
    """Encapsula pré-processamento e classificador em um Pipeline."""
    return Pipeline(
        steps=[
            ("preprocessamento", preprocessador),
            (
                "classificador",
                LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
            ),
        ]
    )


def buscar_melhor_c(pipeline, X_train, y_train):
    """Otimiza o C por validação cruzada de 5 folds em uma grade fixa."""
    grade = {"classificador__C": [0.01, 0.1, 1, 10, 100]}
    busca = GridSearchCV(
        pipeline, param_grid=grade, cv=5, scoring="accuracy", n_jobs=-1
    )
    busca.fit(X_train, y_train)
    return busca


def main():
    df = carregar_dados(CAMINHO_DADOS)
    X = df.drop(columns="churn")
    y = df["churn"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    print(f"Treino: {X_train.shape[0]} linhas | Teste: {X_test.shape[0]} linhas")
    print(f"Churn no treino: {y_train.mean():.2%} | Churn no teste: {y_test.mean():.2%}")

    pipeline = construir_pipeline(construir_preprocessador())

    busca = buscar_melhor_c(pipeline, X_train, y_train)
    print(f"Melhor C: {busca.best_params_['classificador__C']}")
    print(f"Acurácia média na validação cruzada: {busca.best_score_:.4f}")

    # refit=True (padrão do GridSearchCV): o melhor estimador já foi
    # reajustado com todo o X_train, então é o modelo final
    modelo_final = busca.best_estimator_
    y_pred = modelo_final.predict(X_test)
    print(f"Acurácia no teste: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))

    joblib.dump(modelo_final, CAMINHO_MODELO)
    print(f"Modelo serializado em {CAMINHO_MODELO}")


if __name__ == "__main__":
    main()

# PARÁGRAFO FINAL: POR QUE ESTA ORGANIZAÇÃO É MAIS SEGURA PARA PRODUÇÃO
# Transformar os dados manualmente antes de treinar exige repetir os mesmos
# passos, na mesma ordem e com os mesmos parâmetros, toda vez que um lote
# novo chega para previsão. Qualquer esquecimento ou divergência (um scaler
# reajustado em dados novos, um encoder treinado em outro conjunto de
# categorias, uma coluna fora de ordem) muda silenciosamente o significado
# das features e corrompe as previsões. O Pipeline torna o pré-processamento
# parte do próprio modelo: tudo o que foi aprendido no treino fica
# serializado junto com o classificador em um único artefato, e a produção
# só precisa chamar predict sobre dados crus. Assim, o caminho percorrido
# pelos dados em produção é exatamente o mesmo do treino, o que elimina por
# construção o risco de data leakage e de inconsistência entre ambientes.

Overwriting pipeline_final_churn.py


In [13]:
%run pipeline_final_churn.py

Treino: 9600 linhas | Teste: 2400 linhas
Churn no treino: 17.56% | Churn no teste: 17.54%


Melhor C: 0.1
Acurácia média na validação cruzada: 0.8247
Acurácia no teste: 0.8250
              precision    recall  f1-score   support

           0       0.83      1.00      0.90      1979
           1       0.56      0.01      0.02       421

    accuracy                           0.82      2400
   macro avg       0.69      0.50      0.46      2400
weighted avg       0.78      0.82      0.75      2400

Modelo serializado em pipeline_final_churn.joblib


In [14]:
import os

arquivos = sorted(nome for nome in os.listdir(".") if os.path.isfile(nome))
for nome in arquivos:
    print(f"{nome} ({os.path.getsize(nome) / 1024:.1f} KB)")

assinantes_neostream.csv (604.1 KB)
pipeline_classificacao.py (0.6 KB)
pipeline_final_churn.joblib (4.0 KB)
pipeline_final_churn.py (5.5 KB)
preprocessamento_heterogeneo.py (1.2 KB)
renefrancisco_ferreirasantos_DR3_TP1.ipynb (37.9 KB)


# Conclusão da entrega

O ponto central deste TP atravessa todos os exercícios: nenhuma transformação é aprendida antes da divisão estratificada e nenhuma etapa de pré-processamento vive fora do `Pipeline`. Com isso, a avaliação no teste simula fielmente a chegada de dados novos em produção, e o artefato serializado carrega consigo tudo o que o treino aprendeu, pronto para receber lotes novos apenas com `predict` sobre dados crus.

**Observação de engenharia:** a acurácia de 82,5% é puxada pela classe majoritária: o `classification_report` do Exercício 8 mostra que, com o limiar padrão de 0,5, o modelo quase não prevê a classe 1 (recall de churn baixo), o que é esperado em uma base com 17,6% de churn e um sinal fraco. Para o modelo de produção, os próximos passos naturais são calibrar o limiar de decisão, comparar `class_weight="balanced"` ou trocar a métrica da busca por F1/`average_precision`. Nenhuma dessas mudanças altera a estrutura de preparação de dados construída neste TP, que continua válida em qualquer um desses cenários.